In [1]:
import sys
from pathlib import Path
import os
import json

project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from tqdm import tqdm

from Utils.DataUtils import build_ae_dataloaders
from Models.AutoEncoder import AutoEncoder
from Utils.FeatureUtils import extract_features

from xgboost import XGBClassifier
from sklearn.metrics import classification_report
from sklearn.metrics import f1_score, fbeta_score, recall_score, precision_score, roc_auc_score, average_precision_score, accuracy_score

In [2]:
# Config
BATCH_SIZE = 256
LR = 1e-3
EPOCHS = 30
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
# AE training, train only non-fraud
train_loader_ae, val_loader, test_loader, input_dim = build_ae_dataloaders(
    batch_size=BATCH_SIZE,
    mode="ae",
    train_only_nonfraud=True,
    return_labels=True
)

# Train fraud + non-fraud
train_loader_full, _, _, _ = build_ae_dataloaders(
    batch_size=BATCH_SIZE,
    mode="ae",
    train_only_nonfraud=False,
    return_labels=True
)

print("Input dim:", input_dim)

[INFO] Project root: C:\DeepLearning\DL_project
[INFO] Data dir: C:\DeepLearning\DL_project\data
[INFO] Loading train data from: C:\DeepLearning\DL_project\data\train_transaction.csv
[INFO] Loading test data from: C:\DeepLearning\DL_project\data\test_transaction.csv
[INFO] Train shape: (590540, 394)
[INFO] Test shape: (506691, 393)


C:\DeepLearning\DL_project\Utils\Preprocess.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col + "_missing"] = df[col].isna().astype(int)
C:\DeepLearning\DL_project\Utils\Preprocess.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col + "_missing"] = df[col].isna().astype(int)
C:\DeepLearning\DL_project\Utils\Preprocess.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using

[INFO] Train: (455902, 776)
[INFO] Val: (59054, 776)
[INFO] Test: (59054, 776)
[INFO] Project root: C:\DeepLearning\DL_project
[INFO] Data dir: C:\DeepLearning\DL_project\data
[INFO] Loading train data from: C:\DeepLearning\DL_project\data\train_transaction.csv
[INFO] Loading test data from: C:\DeepLearning\DL_project\data\test_transaction.csv
[INFO] Train shape: (590540, 394)
[INFO] Test shape: (506691, 393)


C:\DeepLearning\DL_project\Utils\Preprocess.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col + "_missing"] = df[col].isna().astype(int)
C:\DeepLearning\DL_project\Utils\Preprocess.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col + "_missing"] = df[col].isna().astype(int)
C:\DeepLearning\DL_project\Utils\Preprocess.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using

[INFO] Train: (472432, 776)
[INFO] Val: (59054, 776)
[INFO] Test: (59054, 776)
Input dim: 776


In [4]:
# Load AutoEncoder checkpoint
checkpoint = torch.load("checkpoints/autoencoder.pt", map_location=DEVICE)

model = AutoEncoder(
    input_dim=checkpoint["input_dim"],
    latent_dim=checkpoint["latent_dim"],
    hidden_dims=checkpoint["hidden_dims"],
).to(DEVICE)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

# Extract Features
X_train, y_train = extract_features(model, train_loader_full, DEVICE)
X_val, y_val = extract_features(model, val_loader, DEVICE)
X_test, y_test = extract_features(model, test_loader, DEVICE)
print("Shapes:", X_train.shape, X_val.shape, X_test.shape)

# Train distribution
print("Train distribution:")
n_pos = (y_train == 1).sum()
n_neg = (y_train == 0).sum()
scale_pos_weight = n_neg / n_pos

print("0:", n_neg)
print("1:", n_pos)
print("scale_pos_weight:", scale_pos_weight)

Shapes: (472432, 1037) (59054, 1037) (59054, 1037)
Train distribution:
0: 455902
1: 16530
scale_pos_weight: 27.580278281911674


In [9]:
# Parameters
param_grid = {
    "iterations": [2000, 3000, 4000, 5000],
    "depth": [8, 10],
    "learning_rate": [0.025, 0.03]
}

# Train XGBoost
def train_and_eval(params):
    model_xgb = XGBClassifier(
        n_estimators=params["iterations"],
        max_depth=params["depth"],
        learning_rate=params["learning_rate"],
        objective="binary:logistic",
        eval_metric="aucpr",
        scale_pos_weight=scale_pos_weight,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=10,
        random_state=42,
        early_stopping_rounds=100,
        n_jobs=-1
    )
    model_xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=100)

    # Validation probabilities
    y_proba_val = model_xgb.predict_proba(X_val)[:, 1]

    # Best Threshold (only on validation)
    best_f2 = -1
    best_thresh = 0.5

    for t in np.arange(0.01, 0.99, 0.005):
        y_pred = (y_proba_val >= t).astype(int)
        f2 = fbeta_score(y_val, y_pred, beta=2)
        if f2 > best_f2:
            best_f2 = f2
            best_thresh = t
    return model_xgb, best_f2, best_thresh

In [10]:
# Grid search
best_val_f2 = -1
best_model = None
best_params = None
best_threshold = None

for it in param_grid["iterations"]:
    for d in param_grid["depth"]:
        for lr in param_grid["learning_rate"]:
            params = {
                "iterations": it,
                "depth": d,
                "learning_rate": lr
            }
            print(f"Training: {params}")
            model_xgb, f2, thresh = train_and_eval(params)
            print(f"Validation F2: {f2:.4f} | threshold: {thresh:.3f}")
            if f2 > best_val_f2:
                best_val_f2 = f2
                best_model = model_xgb
                best_params = params
                best_threshold = thresh

# Save best model
best_model.save_model("checkpoints/best_xgboost.json")
best_config = {
    "best_val_f2": float(best_val_f2),
    "best_threshold": float(best_threshold),
    "params": best_params
}
with open("checkpoints/best_xgboost_params.json", "w") as f:
    json.dump(best_config, f, indent=4)

print("Saved best xgboost model + parameters")

Training: {'iterations': 2000, 'depth': 8, 'learning_rate': 0.025}
[0]	validation_0-aucpr:0.35804
[100]	validation_0-aucpr:0.59702
[200]	validation_0-aucpr:0.64953
[300]	validation_0-aucpr:0.67693
[400]	validation_0-aucpr:0.69549
[500]	validation_0-aucpr:0.70920
[600]	validation_0-aucpr:0.72015
[700]	validation_0-aucpr:0.72822
[800]	validation_0-aucpr:0.73447
[900]	validation_0-aucpr:0.74054
[1000]	validation_0-aucpr:0.74556
[1100]	validation_0-aucpr:0.75019
[1200]	validation_0-aucpr:0.75448
[1300]	validation_0-aucpr:0.75855
[1400]	validation_0-aucpr:0.76221
[1500]	validation_0-aucpr:0.76538
[1600]	validation_0-aucpr:0.76875
[1700]	validation_0-aucpr:0.77157
[1800]	validation_0-aucpr:0.77421
[1900]	validation_0-aucpr:0.77650
[1999]	validation_0-aucpr:0.77877
Validation F2: 0.7324 | threshold: 0.570
Training: {'iterations': 2000, 'depth': 8, 'learning_rate': 0.03}
[0]	validation_0-aucpr:0.35804
[100]	validation_0-aucpr:0.60856
[200]	validation_0-aucpr:0.66212
[300]	validation_0-aucpr:0.

In [11]:
# Final Test Evaluation
print("\n=== TEST EVALUATION ===")

# Use best model
y_proba_test = best_model.predict_proba(X_test)[:, 1]

# Default threshold (0.5)
y_pred_default = (y_proba_test >= 0.5).astype(int)

print("=== DEFAULT THRESHOLD (0.5) ===")
print(classification_report(y_test, y_pred_default))
print("ROC_AUC:", roc_auc_score(y_test, y_proba_test))
print("PR_AUC:", average_precision_score(y_test, y_proba_test))
print("ACCURACY:", accuracy_score(y_test, y_pred_default))
print("PRECISION:", precision_score(y_test, y_pred_default))
print("RECALL:", recall_score(y_test, y_pred_default))
print("F1:", f1_score(y_test, y_pred_default))
print("F2:", fbeta_score(y_test, y_pred_default, beta=2))


=== TEST EVALUATION ===
=== DEFAULT THRESHOLD (0.5) ===
              precision    recall  f1-score   support

         0.0       0.99      1.00      0.99     56988
         1.0       0.89      0.66      0.76      2066

    accuracy                           0.99     59054
   macro avg       0.94      0.83      0.87     59054
weighted avg       0.98      0.99      0.98     59054

ROC_AUC: 0.9575848613634527
PR_AUC: 0.8019637909444124
ACCURACY: 0.9851999864530768
PRECISION: 0.8921052631578947
RECALL: 0.6563407550822846
F1: 0.7562744004461796
F2: 0.6929681112019623


In [13]:
# Evaluation: Best Threshold (From Validation)
y_pred_final = (y_proba_test >= best_threshold).astype(int)

# Final evaluation
print("\n=== FINAL RESULTS (OPTIMISED THRESHOLD) ===")
print("Threshold:", best_threshold)
print("ACCURACY:", accuracy_score(y_test, y_pred_final))
print("PRECISION:", precision_score(y_test, y_pred_final))
print("RECALL:", recall_score(y_test, y_pred_final))
print("F1:", f1_score(y_test, y_pred_final))
print("F2:", fbeta_score(y_test, y_pred_final, beta=2))
print(classification_report(y_test, y_pred_final))

# Save threshold + metadata
checkpoint = {
    "model_type": "xgboost",
    "best_threshold": float(best_threshold),
    "best_val_f2": float(best_val_f2),
    "test_roc_auc": float(roc_auc_score(y_test, y_proba_test)),
    "test_pr_auc": float(average_precision_score(y_test, y_proba_test)),
    "test_accuracy": float(accuracy_score(y_test, y_pred_final)),
    "test_precision": float(precision_score(y_test, y_pred_final)),
    "test_recall": float(recall_score(y_test, y_pred_final)),
    "test_f1": float(f1_score(y_test, y_pred_final)),
    "test_f2": float(fbeta_score(y_test, y_pred_final, beta=2)),
    "input_dim": int(input_dim), # metadata
    "latent_dim": 256,
    "scale_pos_weight": float(scale_pos_weight),
    "features_dim": int(X_train.shape[1])
}

with open("checkpoints/xgboost_data_checkpoint.json", "w") as f:
    json.dump(checkpoint, f, indent=4)

print("All checkpoints saved!")


=== FINAL RESULTS (OPTIMISED THRESHOLD) ===
Threshold: 0.09499999999999999
ACCURACY: 0.9776475767941206
PRECISION: 0.6551580698835274
RECALL: 0.7623426911907066
F1: 0.7046979865771812
F2: 0.7381889763779528
              precision    recall  f1-score   support

         0.0       0.99      0.99      0.99     56988
         1.0       0.66      0.76      0.70      2066

    accuracy                           0.98     59054
   macro avg       0.82      0.87      0.85     59054
weighted avg       0.98      0.98      0.98     59054

All checkpoints saved!
